# Sparkle V2 — Fase 4: Tratamiento de datos faltantes

Este notebook asume que ya corriste la Fase 2+3 (features extraídos en `.npz` para Train/Validation/Test en Drive).

**Qué hace:**
1. Carga todos los `.npz` de los tres splits.
2. Para cada secuencia, interpola linealmente los frames con NaN (gaps cortos causados por detección fallida puntual).
3. Descarta clips que superen un umbral de NaN (frames sin rostro detectado) — no conviene inventar datos para tramos largos.
4. Guarda un dataset consolidado en un único archivo por split (`.npz` con arrays apilados), listo para alimentar directamente el LSTM en la Fase 5/6.

Con los resultados que ya tenemos (0.02-0.07% NaN promedio, solo 1 clip con >30% en todo el dataset), este paso es rápido — pero lo dejamos como función reutilizable y documentada, coherente con el Cap. 2.1.12.2 de la tesis.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

import os, glob
import numpy as np
import pandas as pd

PROJECT_DIR = '/content/drive/MyDrive/SparkleV2'
FEATURES_DIR = f'{PROJECT_DIR}/features'
DATASET_DIR = f'{PROJECT_DIR}/dataset_final'
os.makedirs(DATASET_DIR, exist_ok=True)

FEATURE_NAMES = ['ear_avg', 'ear_left', 'ear_right', 'yaw', 'pitch']

# Umbral de descarte: si más de este % de frames de un clip no tienen rostro detectado, se descarta
MAX_NAN_THRESHOLD_PCT = 30.0

print('Listo.')

Mounted at /content/drive
Listo.


## 1. Función de interpolación

Interpolación lineal por columna (cada feature se interpola de forma independiente). Si el primer o último frame de la secuencia es NaN (no hay frame previo/posterior del cual interpolar), se rellena con el valor válido más cercano (forward/backward fill) en vez de dejarlo vacío.

In [3]:
def interpolate_sequence(seq):
    """seq: array (n_frames, n_features) con posibles NaN. Devuelve una copia interpolada."""
    seq = seq.copy()
    n_frames, n_features = seq.shape
    x = np.arange(n_frames)

    for f in range(n_features):
        col = seq[:, f]
        valid = ~np.isnan(col)

        if valid.sum() == 0:
            # No hay ni un solo frame válido en toda la columna: no se puede interpolar
            continue
        if valid.sum() == n_frames:
            # No hay NaN, no hace falta tocar nada
            continue

        # Interpolación lineal en los huecos internos
        col_interp = np.interp(x, x[valid], col[valid])
        seq[:, f] = col_interp

    return seq

## 2. Procesar cada split: cargar, interpolar, filtrar y consolidar

In [4]:
def build_split_dataset(split):
    split_dir = f'{FEATURES_DIR}/{split}'
    npz_files = sorted(glob.glob(f'{split_dir}/*.npz'))

    sequences = []
    clip_ids = []
    engagement_labels = []
    boredom_labels = []
    confusion_labels = []
    frustration_labels = []

    discarded_high_nan = 0
    discarded_all_nan_feature = 0

    for npz_path in npz_files:
        data = np.load(npz_path, allow_pickle=True)
        seq = data['sequence']

        pct_nan_frames = np.isnan(seq).any(axis=1).mean() * 100
        if pct_nan_frames > MAX_NAN_THRESHOLD_PCT:
            discarded_high_nan += 1
            continue

        seq_interp = interpolate_sequence(seq)

        # Si después de interpolar todavía queda algún NaN (columna 100% vacía), se descarta
        if np.isnan(seq_interp).any():
            discarded_all_nan_feature += 1
            continue

        clip_id = os.path.splitext(os.path.basename(npz_path))[0]
        sequences.append(seq_interp)
        clip_ids.append(clip_id)
        engagement_labels.append(int(data['engagement']))
        boredom_labels.append(int(data['boredom']))
        confusion_labels.append(int(data['confusion']))
        frustration_labels.append(int(data['frustration']))

    X = np.stack(sequences)  # (n_clips, n_frames, n_features)

    print(f'{split}: {len(sequences)} clips retenidos | '
          f'{discarded_high_nan} descartados por >{MAX_NAN_THRESHOLD_PCT}% NaN | '
          f'{discarded_all_nan_feature} descartados por feature 100% vacía')

    return {
        'X': X,
        'clip_ids': np.array(clip_ids),
        'engagement': np.array(engagement_labels),
        'boredom': np.array(boredom_labels),
        'confusion': np.array(confusion_labels),
        'frustration': np.array(frustration_labels),
        'feature_names': FEATURE_NAMES,
    }

In [5]:
datasets = {}
for split in ['Train', 'Validation', 'Test']:
    datasets[split] = build_split_dataset(split)

Train: 4851 clips retenidos | 1 descartados por >30.0% NaN | 0 descartados por feature 100% vacía
Validation: 1429 clips retenidos | 0 descartados por >30.0% NaN | 0 descartados por feature 100% vacía
Test: 1638 clips retenidos | 0 descartados por >30.0% NaN | 0 descartados por feature 100% vacía


## 3. Guardar el dataset consolidado en Drive

Un `.npz` por split, con todo apilado. Esto es lo que va a leer directamente el notebook de la Fase 5/6 (armado de tensores + entrenamiento del LSTM), sin tener que volver a tocar los ~7900 archivos individuales.

In [6]:
for split, d in datasets.items():
    out_path = f'{DATASET_DIR}/{split.lower()}_consolidado.npz'
    np.savez_compressed(
        out_path,
        X=d['X'],
        clip_ids=d['clip_ids'],
        engagement=d['engagement'],
        boredom=d['boredom'],
        confusion=d['confusion'],
        frustration=d['frustration'],
        feature_names=d['feature_names'],
    )
    print(f'Guardado: {out_path} — shape X: {d["X"].shape}')

Guardado: /content/drive/MyDrive/SparkleV2/dataset_final/train_consolidado.npz — shape X: (4851, 60, 5)
Guardado: /content/drive/MyDrive/SparkleV2/dataset_final/validation_consolidado.npz — shape X: (1429, 60, 5)
Guardado: /content/drive/MyDrive/SparkleV2/dataset_final/test_consolidado.npz — shape X: (1638, 60, 5)


## 4. Verificación final: distribución de clases

Chequeo rápido de balance de clases en Engagement y Boredom por split — importante para la Fase 5/6, porque si está muy desbalanceado (algo típico en DAiSEE, ya lo mencionás en tu tesis) hay que decidir si se usa class weighting o resampling al entrenar.

In [7]:
for split, d in datasets.items():
    print(f'--- {split} ---')
    print('Engagement:', pd.Series(d['engagement']).value_counts().sort_index().to_dict())
    print('Boredom:   ', pd.Series(d['boredom']).value_counts().sort_index().to_dict())
    print()

--- Train ---
Engagement: {0: 33, 1: 200, 2: 2474, 3: 2144}
Boredom:    {0: 2181, 1: 1479, 2: 1043, 3: 148}

--- Validation ---
Engagement: {0: 23, 1: 143, 2: 813, 3: 450}
Boredom:    {0: 446, 1: 376, 2: 475, 3: 132}

--- Test ---
Engagement: {0: 4, 1: 81, 2: 849, 3: 704}
Boredom:    {0: 747, 1: 519, 2: 335, 3: 37}



## Checkpoint de la Fase 4

Pegame acá el output completo de las secciones 2 y 4 (cuántos clips se retuvieron/descartaron por split, y la distribución de clases de Engagement y Boredom en cada uno).

En paralelo, cuando estén vos y tu amigo, definimos la Fase 5 (qué combinación de labels usar como target — Opción A/B/C que discutimos) y armamos el notebook de entrenamiento del LSTM directamente sobre estos archivos consolidados.

In [8]:
import pandas as pd

train_data = np.load('/content/drive/MyDrive/SparkleV2/dataset_final/train_consolidado.npz', allow_pickle=True)
crosstab = pd.crosstab(train_data['engagement'], train_data['boredom'],
                        rownames=['Engagement'], colnames=['Boredom'])
print(crosstab)
print('\nCorrelación Engagement-Boredom:', pd.Series(train_data['engagement']).corr(pd.Series(train_data['boredom'])))

Boredom        0    1    2   3
Engagement                    
0              5    2    8  18
1             22   35  109  34
2            764  903  731  76
3           1390  539  195  20

Correlación Engagement-Boredom: -0.41920193668152406
